In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
from intecomm_analytics.dataframes.main_1858_to_stata import to_stata, variable_labels
from intecomm_analytics.dataframes import get_df_main_1858

In [ ]:
df_main = get_df_main_1858(analysis_folder, fasting_hours=8.0)

In [ ]:
# check variable_labels for 80 char limit
varlabels = variable_labels()
# export to stata
to_stata(df_main, analysis_folder)

In [ ]:
# export a csv or counts for the primary variables, if needed
df = df_main.groupby(by=["primary_cohort_str"]).agg({col:"count" for col in df_main.columns if col.startswith("primary")})
df = pd.concat([df, pd.DataFrame([df.sum()])])
df = df.reset_index()

df_new = (
    df.melt(id_vars='index', var_name='variable', value_name='value')
    .pivot(index='variable', columns='index', values='value')
    .reset_index().reset_index(drop=True)
    .rename(columns={0:"TOTALS"})
)
df_new